# DeepSeekMoE experiments

This notebook measures routing balance, active-parameter trade-offs, the shared-expert ablation, and isolated dense-versus-MoE latency. The measurements are intentionally small and reproducible.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch

from src.experiments.analyze_routing import (
    inspect_model_routing,
    moe_parameter_report,
)
from src.experiments.benchmark_ffn import compare_ffn
from src.model import DeepSeekConfig, DeepSeekMoE, TinyDeepSeek

torch.manual_seed(20)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Routing diagnostics

Hooks observe every routed layer during one evaluation pass. They do not update router biases or model weights.

In [ ]:
config = DeepSeekConfig(
    vocab_size=512,
    max_seq_len=32,
    dim=64,
    n_layers=3,
    n_dense_layers=1,
    n_heads=4,
    q_lora_rank=16,
    kv_lora_rank=8,
    qk_nope_head_dim=8,
    qk_rope_head_dim=4,
    v_head_dim=8,
    inter_dim=128,
    moe_inter_dim=32,
    n_routed_experts=4,
    n_activated_experts=2,
)
model = TinyDeepSeek(config).to(device).eval()
tokens = torch.randint(0, config.vocab_size, (4, 32), device=device)
routing = inspect_model_routing(model, tokens)
for name, metrics in routing.items():
    print(name, metrics)

## Total versus active parameters

Changing `n_activated_experts` changes the computation on a token path, not the stored routed-expert bank.

In [ ]:
for top_k in (1, 2, 4):
    sweep_config = DeepSeekConfig(
        dim=64,
        n_heads=4,
        q_lora_rank=16,
        kv_lora_rank=8,
        qk_nope_head_dim=8,
        qk_rope_head_dim=4,
        v_head_dim=8,
        n_routed_experts=4,
        n_activated_experts=top_k,
        moe_inter_dim=32,
    )
    report = moe_parameter_report(sweep_config)
    print(top_k, report)

## Shared-expert ablation

The zero-shared configuration is valid for an ablation. The default configuration still creates and runs one shared expert.

In [ ]:
for shared, top_k in ((1, 2), (0, 3)):
    ablation_config = DeepSeekConfig(
        dim=64,
        n_heads=4,
        q_lora_rank=16,
        kv_lora_rank=8,
        qk_nope_head_dim=8,
        qk_rope_head_dim=4,
        v_head_dim=8,
        n_routed_experts=4,
        n_shared_experts=shared,
        n_activated_experts=top_k,
        moe_inter_dim=32,
    )
    layer = DeepSeekMoE(ablation_config)
    print({'shared': shared, 'top_k': top_k, 'parameters': sum(p.numel() for p in layer.parameters())})

## Isolated FFN benchmark

The benchmark includes warmup and reports the median latency. Increase the sequence length and batch size to see how token dispatch changes the result.

In [ ]:
benchmark_config = DeepSeekConfig(
    dim=64,
    n_heads=4,
    q_lora_rank=16,
    kv_lora_rank=8,
    qk_nope_head_dim=8,
    qk_rope_head_dim=4,
    v_head_dim=8,
    inter_dim=192,
    moe_inter_dim=64,
    n_routed_experts=4,
    n_activated_experts=2,
)
inputs = torch.randn(2, 32, benchmark_config.dim, device=device)
compare_ffn(benchmark_config, inputs, warmup=2, repeats=5)